In [11]:
from typing import Dict, List, Set
from pathlib import Path
import subprocess
import json
import networkx as nx

project_root = Path('../../..').resolve()  # Adjust as needed to find the project root
legacy_root = project_root / '../legacy'

src_dir = legacy_root / 'src'
temp_dir = project_root / 'tmp'
code_graph_gml = project_root / 'artifacts/code_graph.gml'

print(f"Project root: {project_root}")

Project root: /Users/QTE2333/repos/legacy-projects


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# get the list of source and header files used in compilation

def get_headers_from_bear_entry(entry: Dict[str, str|List[str]]) -> List[Path]:
    source_file = entry["file"]
    workdir = Path(entry["directory"])
    args = entry["arguments"]

    # Only keep preprocessing-relevant arguments
    allowed_prefixes = ("-I", "-isystem", "-D", "-U", "-include", "-std=")
    filtered_args = [
        arg for arg in args[1:]  # skip compiler
        if arg.startswith(allowed_prefixes)
    ]
    include_dirs = [Path(arg[2:]) for arg in args if arg.startswith(("-I"))]
    include_dirs = [d for d in include_dirs if d.is_relative_to(workdir)]

    cmd = ["clang++", "-M", "-fsyntax-only", *filtered_args, source_file]

    try:
        output = subprocess.check_output(cmd, cwd=workdir, stderr=subprocess.DEVNULL, text=True)
        # Output is a Makefile-style dependency line: target: file1 file2 ...
        # Extract the right-hand side and split
        deps_line = output.split(":", 1)[1]
        headers = [Path(h) for h in deps_line.replace("\\", "").strip().split()]
        headers = [h for h in headers if not h.is_absolute()]
        headers = [(workdir / h).resolve() for h in headers]
        return headers
    except subprocess.CalledProcessError as e:
        print(f"Clang failed on {source_file}: {e}")
        return []
    except IndexError:
        print(f"Malformed dependency output for {source_file}")
        return []

if False:
    with open(src_dir / 'compile_commands.json', 'r', encoding='utf-8') as f:
        compile_commands = json.load(f)

    source_files = set(Path(m['file']) for m in compile_commands)
    header_files = set(Path(h) for m in compile_commands for h in get_headers_from_bear_entry(m))

    # find source and header files that are not used in compilation
    found_sources = [f for f in src_dir.glob('**/*.c*')]
    found_headers = [f for f in src_dir.glob('**/*.h*')]
    unused_sources = [f for f in found_sources if f not in source_files]
    unused_headers = [f for f in found_headers if f not in header_files]
    print(f"Source files: found: {len(found_sources)}, used: {len(source_files)}, unused: {len([f for f in found_sources if f not in source_files])}")
    print(f"Header files: found: {len(found_headers)}, used: {len(header_files)}, unused: {len([f for f in found_headers if f not in header_files])}")

    if len(unused_sources):
        print("Unused source files:")
        for f in unused_sources:
            print(f"  {f.relative_to(src_dir)}")
    if len(unused_headers):
        print("Unused header files:")
        for f in unused_headers:
            print(f"  {f.relative_to(src_dir)}")

In [12]:
# ensure Doxyfile has these fields set:
'''
OUTPUT_DIRECTORY       = doxygen_output
EXTRACT_ALL            = YES
EXTRACT_PRIVATE        = YES
EXTRACT_STATIC         = YES
EXTRACT_ANON_NSPACES   = YES
INPUT                  = src/
FILE_PATTERNS          = *.cpp *.cc *.c *.h *.hh *.hpp
RECURSIVE              = YES
GENERATE_XML           = YES
MACRO_EXPANSION        = YES
EXPAND_ONLY_PREDEF     = YES
PREDEFINED             = args(list)=list \
                         DECLARE_SPEC_FUN(fun)=SPEC_FUN fun \
                         DECLARE_SPELL_FUN(fun)=SPELL_FUN fun \
                         DECLARE_DO_FUN(fun)=DO_FUN fun
HAVE_DOT               = NO # only for graph visualization in html (turn on to enable other graph features)
CALL_GRAPH             = YES # only for graph visualization in html (requires HAVE_DOT)
CALLER_GRAPH           = YES # only for graph visualization in html (requires HAVE_DOT)
DOT_MULTI_TARGETS      = YES # only for graph visualization in html (requires HAVE_DOT)
'''

doxygen_output = legacy_root / 'doxygen_output'

In [10]:
doxygen_output.resolve()

PosixPath('/Users/QTE2333/repos/legacy/doxygen_output')

In [13]:
import legacy_common.doxygen_parse as dp
import legacy_common.doxygen_graph as dg

db = dp.EntityDatabase.from_xml_dir(doxygen_output / "xml")
dp.save_db(db, code_graph_gml.with_suffix('.json'))

# start building a list of compound or member IDs to exclude from the graph
EXCLUDE_GIDS = []

2026-03-16 23:50:52.954 | INFO     | legacy_common.doxygen_parse:save_db:571 - Database saved to /Users/QTE2333/repos/legacy-projects/artifacts/code_graph.json


In [6]:
for eid, m in db.members.items():
    if m.name.startswith("__"):
        EXCLUDE_GIDS.append(eid.member)

In [14]:
# TEST
# make sure that all related member IDs agree on a single body location 
# (could be multiple decl or file locs) and agree on a kind, name, and signature

for mid, members in db.member_groups.items():
    # Check if all members agree on a single body location
    # Check if all members agree on a kind, name, and signature
    body_locs = {str(m.body) for m in members}
    kinds = {m.kind for m in members}
    names = {m.name for m in members}
    signatures = {m.signature for m in members}
    if len(body_locs) != 1:
        print(f"Member {mid} has inconsistent body locations: {body_locs}")
    if len(kinds) != 1:
        print(f"Member {mid} has inconsistent kinds: {kinds}")
    if len(names) != 1:
        print(f"Member {mid} has inconsistent names: {names}")
    if len(signatures) != 1:
        print(f"Member {mid} has inconsistent signatures: {signatures}")

In [15]:
# TEST
# find all the functions that have the same name but different signature, helps to identify unintentional overloads
fn_names = {}
for eid, m in db.members.items():
    if m.kind != 'function':
        continue
    fn_names.setdefault(m.name, {}).setdefault(m.signature, set()).add(str(eid))

for name, sigs in fn_names.items():
    if len(sigs) > 1:
        print(f"Function '{name}' has multiple signatures:\n{'\n'.join(f'  {sig} [{', '.join(eids)}]' for sig, eids in sigs.items())}")
        pass


Function 'RoomID' has multiple signatures:
  RoomID::RoomID() [class_room_i_d_1acb9fb86413bad3f692989a4d59200b39]
  RoomID::RoomID(const Vnum &vnum, int number=1) [class_room_i_d_1a3e79f4e25688dcae55a3e87eadec58c3]
  RoomID::RoomID(const String &s) [class_room_i_d_1a8e94af33f120368add8456779bd45624]
  RoomID::RoomID(const RoomID &id) [class_room_i_d_1a87243b34889175349bfb7e136a110236]
Function 'is_valid' has multiple signatures:
  bool RoomID::is_valid() const [class_room_i_d_1a22b45ee004bb3d6f0c04f41d37890349]
  bool worldmap::Coordinate::is_valid() const [classworldmap_1_1_coordinate_1a00c2e6fd32e3a3f6b3878323ffe89b36]
  bool Location::is_valid() const [class_location_1a24f501f6130736146598fe5823181d11]
Function 'to_int' has multiple signatures:
  int RoomID::to_int() const [class_room_i_d_1a9dcf2728583a256f21c4433cb0944975]
  int worldmap::Coordinate::to_int() const [classworldmap_1_1_coordinate_1a54d856fc97efd81f805bc344781a1d61]
  int Location::to_int() const [class_location_1a0a9

In [16]:
# TEST
# find all function forward declarations so we can remove those that are unnecessary
found_locs = set()
for mid, members in db.member_groups.items():
    body_locs = set()
    decl_locs = set()
    file_locs = set()
    for m in members:
        if m.body:
            body_locs.add((m.body.fn, m.body.line))
        if m.decl:
            decl_locs.add((m.decl.fn, m.decl.line))
        if m.file:
            file_locs.add((m.file.fn, m.file.line))

    body_lines = set((fn, line) for fn, line in body_locs if fn and line)

    for loc in sorted(decl_locs|file_locs, key=lambda x: x[0]):
        if loc not in body_lines:
            found_locs.add(loc)

for loc in sorted(found_locs):
    if loc[0].endswith('.cc'):
        print(f"Declaration location without matching body: {loc}")


Declaration location without matching body: ('src/Game.cc', 14)
Declaration location without matching body: ('src/Note.cc', 49)
Declaration location without matching body: ('src/Note.cc', 50)
Declaration location without matching body: ('src/Note.cc', 51)
Declaration location without matching body: ('src/Room.cc', 30)
Declaration location without matching body: ('src/War.cc', 26)
Declaration location without matching body: ('src/War.cc', 27)
Declaration location without matching body: ('src/War.cc', 28)
Declaration location without matching body: ('src/War.cc', 29)
Declaration location without matching body: ('src/act_info.cc', 79)
Declaration location without matching body: ('src/act_move.cc', 64)
Declaration location without matching body: ('src/act_obj.cc', 72)
Declaration location without matching body: ('src/act_obj.cc', 74)
Declaration location without matching body: ('src/affect/affect_char.cc', 13)
Declaration location without matching body: ('src/affect/affect_obj.cc', 10)
Dec

In [17]:
# TEST
# look at the different kinds of locations for entities, helps to find incorrect or duplicated members
def get_location_kinds(entities: List[dp.DoxygenEntity], exclude_redundant: bool=True):
    body_locs = set()
    decl_locs = set()
    file_locs = set()

    for e in entities:
        if e.body:
            body_locs.add(e.body.id)
        if e.decl:
            decl_locs.add(e.decl.id)

    for e in entities:
        if e.file:
            if exclude_redundant:
                file_line = f"{e.file.fn}:{e.file.line}"
                if any(str(loc_id).startswith(file_line) for loc_id in (decl_locs|body_locs)):
                    continue

            file_locs.add(e.file.id)

    return (entities[0].kind, len(body_locs), len(decl_locs), len(file_locs))

exclude_redundant = True
compounds = [(eid.compound, [compound]) for eid, compound in db.compounds.items()]
member_groups = [(mid, members) for mid, members in db.member_groups.items()]

body_kinds = {}
decl_kinds = {}
file_kinds = {}

for gid, group in (compounds + member_groups):
    kind, body, decl, file = get_location_kinds(group, exclude_redundant)
    for has_loc, loc_set in ((body, body_kinds), (decl, decl_kinds), (file, file_kinds)):
        loc_set.setdefault(kind, {'has': 0, 'total': 0, 'gids': []})
        if has_loc:
            loc_set[kind]['has'] += 1
            loc_set[kind]['gids'].append(gid)
        loc_set[kind]['total'] += 1

all_kinds = set(file_kinds) | set(body_kinds) | set(decl_kinds)
longest = max(len(k) for k in all_kinds) + 13
no_location = set()

print(f"{'file':>{longest}}{'body':>{longest}}{'decl':>{longest}}")
print('-'*longest*3)
for kind in all_kinds:
    if not file_kinds.get(kind, {}).get('has') and not body_kinds.get(kind, {}).get('has') and not decl_kinds.get(kind, {}).get('has'):
        no_location.add(kind)
        continue
#    print(kind, file_kinds.get(kind, {}).get('has', 0), body_kinds.get(kind, {}).get('has', 0), decl_kinds.get(kind, {}).get('has', 0))
    file_kind = f"{kind.upper() if file_kinds[kind]['has'] == file_kinds[kind]['total'] else kind} ({file_kinds[kind]['has']:>4}/{file_kinds[kind]['total']:>4})" if file_kinds.get(kind, {}).get('has') else ' '
    body_kind = f"{kind.upper() if body_kinds[kind]['has'] == body_kinds[kind]['total'] else kind} ({body_kinds[kind]['has']:>4}/{body_kinds[kind]['total']:>4})" if body_kinds.get(kind, {}).get('has') else ' '
    decl_kind = f"{kind.upper() if decl_kinds[kind]['has'] == decl_kinds[kind]['total'] else kind} ({decl_kinds[kind]['has']:>4}/{decl_kinds[kind]['total']:>4})" if decl_kinds.get(kind, {}).get('has') else ' '
    print(f"{file_kind:>{longest}}{body_kind:>{longest}}{decl_kind:>{longest}}")

print(f"\nNo locations found: {', '.join(no_location)}")

print(set(file_kinds['function']['gids']) - set(body_kinds['function']['gids']))
print(body_kinds['function']['gids'])

# ignore member functions that don't have implementations
#EXCLUDE_GIDS.extend(list(set(file_kinds['function']['gids']) - set(body_kinds['function']['gids'])))

                  file                  body                  decl
------------------------------------------------------------------
  variable (   1/2264)  variable (2263/2264)  variable ( 101/2264)
     class (  39/  56)     CLASS (  56/  56)                      
                          DEFINE (  82/  82)                      
  function ( 273/1964)  function (1869/1964)  function ( 548/1964)
 NAMESPACE (  11/  11)                                            
    struct (  40/  80)    STRUCT (  80/  80)                      
                            ENUM (  11/  11)                      
      file ( 207/ 217)                                            
                         TYPEDEF (   9/   9)                      

No locations found: group, dir
{'1a4eb532b804688071bd61ab924df4441f', '1aad1509ab122b4bb8330af939aa1ecc38', '1a246b1124dcc003fad93d7ea8d0f6851b', '1a4fe50af09ebc8ce9f6536838c3e7495a', '1a7174714aa93bef8e2aa2ab3dffa5a53a', '1a8826850c4f8a2871819713ef61148836', '1

In [18]:
print(EXCLUDE_GIDS)

[]


In [19]:
# conduct some reference sanity checks before building a graph
refs = [{
    'from': str(ref[0]),
    'to': ref[1],
    'tag': ref[2],
} for ref in dg.compose_structural_reference_list(db)]
print(len(refs))

# Check if every 'references' ref is mirrored by a 'referencedby' ref
references = set()
referencedby = set()

for ref in refs:
    if ref['tag'] == 'references':
        references.add((ref['from'], ref['to']))
    elif ref['tag'] == 'referencedby':
        referencedby.add((ref['to'], ref['from']))

print(len(references), "references")
print(len(referencedby), "referencedby")

missing_referencedby = references.difference(referencedby)
missing_references = referencedby.difference(references)
print(len(missing_referencedby), "references without a corresponding referencedby")
print(sorted(missing_referencedby)[:5])
print(len(missing_references), "referencedby without a corresponding references")
print(sorted(missing_references)[:5])


70978
27861 references
26806 referencedby
5671 references without a corresponding referencedby
[('_character_8hh_1a028689e9b7999f5320fc2e105b6f5ce1', '_affect_8hh_1aad112b16044504bf60aff1824f05eb45'), ('_character_8hh_1a028689e9b7999f5320fc2e105b6f5ce1', 'class_character_1a41d4a6966bc3ed554b7314089240fbd2'), ('_character_8hh_1a028689e9b7999f5320fc2e105b6f5ce1', 'class_character_1aa8c40c4eb774259101b7354138ece536'), ('_character_8hh_1a028689e9b7999f5320fc2e105b6f5ce1', 'class_object_1ab336f62524013bae78ac04eb5cf6f179'), ('_character_8hh_1a028689e9b7999f5320fc2e105b6f5ce1', 'class_object_1aeb2fe682fe0f2e83434858d89ff1b225')]
4616 referencedby without a corresponding references
[('_auction_8cc_1abc83c1633eac6abe4dd8c5e0b77b1712', '_auction_8hh_1a2cb8c6afdf6b2d58b154a4da34d16039'), ('_auction_8cc_1abc83c1633eac6abe4dd8c5e0b77b1712', '_character_8hh_1a18c23254cb3e6a8c83a80575e17a42cb'), ('_auction_8cc_1abc83c1633eac6abe4dd8c5e0b77b1712', '_character_8hh_1aaab6db46db89573eee49196d2687b7bc'),

In [21]:
exclude_eids: Set[dp.EntityID] = set()
for gid in EXCLUDE_GIDS:
    if gid in db.member_groups:
        for member in db.member_groups[gid]:
            exclude_eids.add(member.id)
        continue

    eid = dp.EntityID.from_str(gid)
    if eid in db.compounds:
        exclude_eids.add(eid)
    else:
        raise Exception(f"bad GID {gid}")

g = dg.build_graph(db, exclude=exclude_eids)
dg.save_graph(g, code_graph_gml)

num_compound_nodes = sum(1 for n, d in g.nodes(data=True) if d['type'] == dg.EntityType.COMPOUND)
num_member_nodes = sum(1 for n, d in g.nodes(data=True) if d['type'] == dg.EntityType.MEMBER)
num_body_nodes = sum(1 for n, d in g.nodes(data=True) if d['type'] == dg.EntityType.BODY)
num_decl_nodes = sum(1 for n, d in g.nodes(data=True) if d['type'] == dg.EntityType.DECLARATION)
num_file_nodes = sum(1 for n, d in g.nodes(data=True) if d['type'] == dg.EntityType.FILE)
num_contains_edges = sum(1 for u, v, k in g.edges(keys=True) if k == dg.Relationship.CONTAINED_BY)
num_includes_edges = sum(1 for u, v, k in g.edges(keys=True) if k == dg.Relationship.INCLUDES)
num_inherits_edges = sum(1 for u, v, k in g.edges(keys=True) if k == dg.Relationship.INHERITS)
num_calls_edges = sum(1 for u, v, k in g.edges(keys=True) if k == dg.Relationship.CALLS)
num_uses_edges = sum(1 for u, v, k in g.edges(keys=True) if k == dg.Relationship.USES)
num_represented_edges = sum(1 for u, v, k in g.edges(keys=True) if k == dg.Relationship.REPRESENTED_BY)
kinds = set(d.get('kind') or d.get('type') for n, d in g.nodes(data=True))

print(f"Graph has {len(g)} nodes, {len(g.edges())} edges")
print(f"Compound nodes: {num_compound_nodes}, Member nodes: {num_member_nodes}")
print(f"Body nodes: {num_body_nodes}, Declaration nodes: {num_decl_nodes}, File nodes: {num_file_nodes}")
print(f"Contains edges: {num_contains_edges}, Includes edges: {num_includes_edges}, Inherits edges: {num_inherits_edges}")
print(f"Calls edges: {num_calls_edges}, Uses edges: {num_uses_edges}, Represented edges: {num_represented_edges}")
print(f"Node kinds: {', '.join(sorted(kinds))}")

2026-03-16 23:54:15.377 | INFO     | legacy_common.doxygen_graph:save_graph:623 - Graph written to /Users/QTE2333/repos/legacy-projects/artifacts/code_graph.gml


Graph has 14511 nodes, 44502 edges
Compound nodes: 469, Member nodes: 4330
Body nodes: 4370, Declaration nodes: 663, File nodes: 4679
Contains edges: 6569, Includes edges: 1656, Inherits edges: 53
Calls edges: 9943, Uses edges: 16569, Represented edges: 9712
Node kinds: body, class, decl, define, dir, enum, file, function, group, namespace, struct, typedef, variable


In [22]:
# TEST
# we shouldn't have two different member IDs pointing to the same location ID.
# this could be caused by default arguments in a header for an otherwise identical
# signature, for those cases I just made some things inline in headers to fix it.
# alternatively could be consolidated on a case-by-case after graph generation.
for n, d in g.nodes(data=True):
    if d.get('type') in (dg.EntityType.BODY, dg.EntityType.FILE, dg.EntityType.DECLARATION):
        candidates = list(g.successors(n))
        if len(candidates) != 1:
            print(f"Node {n} ({d.get('type')}) has multiple candidates: {candidates}")

In [23]:
# For each entity in db.members, collect sets of decl, file, and body locations
entity_locations = {}

for eid, compound in sorted(db.compounds.items(), key=lambda i: i[1].kind):
    members = [compound]
    mid = eid.compound
#for mid, members in db.member_groups.items():

    decl_locs: Set[dp.LocationID] = set()
    file_locs: Set[dp.LocationID] = set()
    body_locs: Set[dp.LocationID] = set()

    for member in members:
        if member.body:
            body_locs.add(member.body.id)
        if member.decl:
            decl_locs.add(member.decl.id)
    for member in members:
        if member.file and member.file.id not in (decl_locs|body_locs):
            file_line = f"{member.file.fn}:{member.file.line}"
            if any(str(loc_id).startswith(file_line) for loc_id in (decl_locs|body_locs)):
                continue
            file_locs.add(member.file.id)

    if len(file_locs) > 0:
#    if len(decl_locs|body_locs) > 2 or len(file_locs) > 0:
        print(f"{mid:>30} [{members[0].name:>20}]: body: {len(body_locs)}, decl: {len(decl_locs)}, file: {len(file_locs)}")
        for l in body_locs:
            print(f"  [body] {l}")
        for l in decl_locs:
            print(f"  [decl] {l}")
        for l in file_locs:
            print(f"  [file] {l}")


         class_departed_player [      DepartedPlayer]: body: 1, decl: 0, file: 1
  [body] src/include/DepartedPlayer.hh:6-18
  [file] src/include/DepartedPlayer.hh:5:1
          class_exit_prototype [       ExitPrototype]: body: 1, decl: 0, file: 1
  [body] src/include/ExitPrototype.hh:9-23
  [file] src/include/ExitPrototype.hh:8:1
                   class_flags [               Flags]: body: 1, decl: 0, file: 1
  [body] src/include/Flags.hh:10-87
  [file] src/include/Flags.hh:9:1
       class_mob_prog_act_list [      MobProgActList]: body: 1, decl: 0, file: 1
  [body] src/include/MobProgActList.hh:9-23
  [file] src/include/MobProgActList.hh:8:1
        classaffect_1_1_affect [      affect::Affect]: body: 1, decl: 0, file: 1
  [body] src/include/affect/Affect.hh:23-45
  [file] src/include/affect/Affect.hh:22:1
           class_war_1_1_event [          War::Event]: body: 1, decl: 0, file: 1
  [body] src/include/War.hh:28-44
  [file] src/include/War.hh:27:5
                  class_social 

In [24]:
for member_id, members in db.member_groups.items():
    body_locs = set()
    for m in members:
        if m.body and m.file and m.body.fn == m.file.fn and m.body.line == m.file.line:
            body_locs.add(m.id.compound)
    if len(body_locs) > 1:
        print(member_id, body_locs)

In [25]:
# TEST
# make sure our method to distinguish the owning compound of a code body works
if True:
    for eid, c in db.compounds.items():
        if c.kind in ('dir',):
            continue
        owner_id = dg.get_body_eid(db, eid.compound)
        assert owner_id is not None and owner_id in db.entities

    for mid, m in db.member_groups.items():
        # if m.kind in ('dir',):
        #     continue
        owner_id = dg.get_body_eid(db, mid)
        assert owner_id is not None and owner_id in db.entities

In [26]:
print(g.edges('class_object_prototype', 'class_vnum', keys=True))

[('class_object_prototype', '_object_prototype_8hh', <Relationship.CONTAINED_BY: 'contained_by'>, None)]


In [27]:
# TEST
# make sure that all members have a body location
for mid, members in db.member_groups.items():
    source_nodes = {}
    if mid not in g.nodes:
#        print(f"Member {mid} not found in graph")
        continue
    for pred in g.predecessors(mid):
        if g.edges[pred, mid].get("type") == dg.Relationship.REPRESENTED_BY:
            source_nodes.setdefault(g.nodes[pred].get('type'), []).append(pred)

    if len(source_nodes.get('body', [])) == 0:
        print(f"{mid} {members[0].signature}")
#        print(f"{mid}: {source_nodes['decl']}")


ValueError: not enough values to unpack (expected 3, got 2)

We now have a directed graph of high level compound entities (namespaces, classes, files, etc) that wholly own low level entities (functions, variables, definitions) with 'contains' edges, and 'requires' edges for dependencies between the low level entities.  Now identify groups of components that are strongly connected with graph cycles.  This turns out to be maybe unnecessary given our approach, but it's useful for looking at the relationships.  In the future these could be documented as composite modules made up of strongly coupled submodules.

Next we prioritize node traversals for documentation.  Basic idea:

Forward pass - start with the smallest components that have the least (zero) dependencies and build up documentation as we move toward higher level modules.  The idea here is to look at how things work and deduce what they do.

Backward pass - start with the highest level and work backwards, using the higher level docs to add context and enhance the lower level documents.  The idea here is to look at how things are used and deduce what they should provide.